# **Google**

In [4]:
# @title Installs
############################################

# 1. NumPy < 2 fixieren (löst den pandas/pyarrow-Absturz)
%pip install --quiet "numpy<2"

# 2. Google Cloud, ADK & Extras installieren (immer mit Anführungszeichen wegen zsh!)
%pip install --upgrade --quiet "google-cloud-aiplatform[agent_engines,adk,evaluation]"
%pip install --upgrade --quiet google-cloud-secret-manager google-cloud-bigquery "anthropic[vertex]"
%pip install --quiet a2a-sdk google-adk
%pip install --quiet litellm mcp python-dotenv

# 3. Versionen kurz prüfen
%pip show google-adk litellm numpy

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gensim 4.3.3 requires scipy<1.14.0,>=1.7.0, but you have scipy 1.17.1 which is incompatible.
astropy 8.0.1 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
contourpy 1.4.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
openfermion 1.7.1 requires requests~=2.32.2, but you have requests 2.34.2 which is incompatible.
streamlit 1.37.1 requires packaging<25,>=20, but you have packaging 26.3 which is incompatible.
streamlit 1.37.1 requires protobuf<6,>=3.20, but you have protobuf 6.33.6 which is incompatible.
streamlit 1.37.1 requires tenacity<9,>=8.1.0, but you have tenacity 9.1.4 which is incompatible.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to

In [1]:
# @title Imports
############################################

import os, asyncio, json, litellm, warnings, vertexai, google.adk, google.auth, google.auth.transport.requests, time
warnings.filterwarnings('ignore')

from google.cloud import storage
from google import genai
from google.genai import types
from google.genai.types import HttpOptions
from google.genai import types as genai_types
from google.adk.agents import LlmAgent, SequentialAgent, ParallelAgent, Agent
from google.adk.events import Event
from google.adk.tools import google_search, url_context, VertexAiSearchTool, load_memory
from google.adk.sessions import InMemorySessionService, VertexAiSessionService
from google.adk.memory import InMemoryMemoryService, VertexAiMemoryBankService
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner
from typing import List
import pandas as pd
from pydantic import BaseModel
from vertexai import Client, types, agent_engines
from vertexai.preview import reasoning_engines  # Deployment, Tracing and Telemetry
from google.genai.types import (
    CreateBatchJobConfig,
    CreateCachedContentConfig,
    EmbedContentConfig,
    FunctionDeclaration,
    GenerateContentConfig,
    HarmBlockThreshold,
    HarmCategory,
    Part,
    SafetySetting,
    Tool,
)

print("✅ Alle Imports erfolgreich geladen!")

23:13:16 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: module 'lib' has no attribute 'GEN_EMAIL'
23:13:16 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: module 'lib' has no attribute 'GEN_EMAIL'


✅ Alle Imports erfolgreich geladen!


In [2]:
# @title Environmental Variables
############################################

# These tell the underlying google-genai client to use Vertex AI
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "1"
os.environ["GOOGLE_CLOUD_PROJECT"] = "deltorobarba"
os.environ["GOOGLE_CLOUD_LOCATION"] = "us-central1"
# Required by LiteLLM for Vertex AI non-Gemini models
os.environ["VERTEXAI_PROJECT"] = "deltorobarba"
os.environ["VERTEXAI_LOCATION"] = "us-central1"
# Initialize Vertex AI client
client = vertexai.Client(project="deltorobarba", location="us-central1")
PROJECT_ID="deltorobarba"
LOCATION="us-central1"

In [ ]:
import vertexai

PROJECT_ID = "deltorobarba"
LOCATION = "us-central1"

vertexai.init(project=PROJECT_ID, location=LOCATION)
print("✅ Erfolgreich über Application Default Credentials (ADC) verbunden.")

In [ ]:
# @title Connect to Google Cloud Project
############################################

# Path 1: Connect to Google Cloud Project
#from google.colab import auth
#auth.authenticate_user()

# Path 2: Take key from local path

KEY_PATH = "/content/deltorobarba-b0e17a2b88f8.json"
with open(KEY_PATH, "w") as f:
    json.dump(key_data, f)

# Load key and connect to GCP
KEY_PATH = "/content/deltorobarba-b0e17a2b88f8.json"
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = KEY_PATH # Robot identity for Google libraries
vertexai.init(project="deltorobarba", location="us-central1")

!gcloud auth activate-service-account --key-file={KEY_PATH} # Authenticate gcloud CLI for !gcloud commands
!gcloud config set project deltorobarba
print("✅ GCP service account is now active.")

Activated service account credentials for: [622694106880-compute@developer.gserviceaccount.com]


To take a quick anonymous survey, run:
  $ gcloud survey

Updated property [core/project].
✅ GCP service account is now active.


###### 📒 *Agent Pipeline*

In [ ]:
# @title Define Agents and Workflow
############################################

# Vertex AI Search (VAIS)
DATA_STORE_ID = "projects/deltorobarba/locations/global/collections/default_collection/dataStores/deltorobarba-adk_1773523070796"
# Google's out-of-the-box RAG system for information retrieval (for Grounding)
  # Documents are loaded onto Google Cloud Storage bucket (here: 'gs://deltorobarba_adk/')
  # Create new in 'Search app' (Custom search (general) in Vertex AI Search
  # --> Discovery Engine API is underlying engine, includes streamAssist. https://docs.cloud.google.com/generative-ai-app-builder/docs
  # VAIS does (managed): Chunk documents. Choose embedding model. Upload vectors to database. Search that database. Re-rank results.
# https://developers.googleblog.com/building-with-gemini-embedding-2/

# 1. Google Search Agent
web_agent = LlmAgent(
    name='web_researcher',
    model='gemini-3.5-flash',
    description=('Use GoogleSearchTool to find news latest news about Gemini. Return only a list of URLs.'),
    instruction='Search for the latest public news on the topic.',
    tools=[google_search],
    output_key='search_results')

# 2. Vertex AI Search Agent
doc_agent = LlmAgent(
    name='internal_specialist',
    model='gemini-3.5-flash',
    instruction='''Search our internal knowledge base for related documents.
    If the tool returns no results, output EXACTLY: "NO INTERNAL DOCUMENTS FOUND".''',
    tools=[VertexAiSearchTool(data_store_id=DATA_STORE_ID)], # Vertex AI Search with DATA_STORE_ID
    output_key='internal_data')

# 3. Parallel Retrieval Agent
parallel_retrieval = ParallelAgent(
    name="parallel_retrieval",
    sub_agents=[web_agent, doc_agent])

# 4. Web Search Analysis Agent (Processes raw URLs)
search_analyst = LlmAgent(
    name='web_analyst',
    model='gemini-3.5-flash',
    instruction='''Use the url_context tool to read the URLs in {search_results}.
    Extract only technical specifications and performance benchmarks.''',
    tools=[url_context],
    output_key='web_analysis')

# 5. Deep Analysis Agent ("Brain")
analyst = LlmAgent(
    name='analyst',
    #model=deepseek,                   # <----- Be careful with region of 3P models for deployed version
    model='gemini-3.5-flash', # hier eventually Gemini Pro
    description=('Compares Gemini 3 vs GPT-4o.'),
    instruction='''
    Check your memory for any user-specific instructions or past preferences.
    Compare public findings ({web_analysis}) against our private data ({internal_data}).
    Identify unique Gemini 3 features that GPT-4o lacks.
    ''',
    tools=[load_memory],  # Recall information from Vertex AI Memory Services!
    output_key='final_comparison')

# 6. CEO Summarizer Agent
summarizer = LlmAgent(
    name='summarizer',
    model='gemini-3.5-flash',
    description=('Draft concise summary'),
    instruction='''
    You are an Executive Intelligence Analyst.
    Summarize the findings in {final_comparison} into 5 punchy bullets for the CEO.
    Ensure you highlight contradictions between internal and public data.
    Max 2000 characters.
    ''',
    output_key='summary' # not used in deployed version
    )

# 7. Root Orchestrator (Master Flow)
root_agent = SequentialAgent(
    name='researcher',
    # no model needed
    # no instructions needed, it just moves data from 0 -> 1 -> 2 -> 3
    sub_agents=[parallel_retrieval, search_analyst, analyst, summarizer],)

In [ ]:
# @title Deploy Pipeline on Agent Engine (⚠️ --> once only!)
############################################

"""
Permissions: Take the AI Platform Reasoning Engine Service Agent that the agent uses:
'service-892203813305@gcp-sa-aiplatform-re.iam.gserviceaccount.com'.
Go to IAM, enable 'Include Google-provided role grants' and add role: 'Discovery Engine Viewer', 'Vertex AI User' and 'Cloud Trace Agent' (write)
Troubleshooting: https://docs.cloud.google.com/agent-builder/agent-engine/troubleshooting/use

Traceing and Logging:
* Cloud Logging API in GCP project aktivieren
* Telemetry API in GCP project aktivieren
* Add role 'Cloud Trace Agent' (write) to service account
* https://docs.cloud.google.com/agent-builder/agent-engine/manage/tracing
* https://docs.cloud.google.com/agent-builder/agent-engine/deploy
"""

# Create GCS bucket first if you haven't already
STAGING_BUCKET = "gs://deltorobarba-agent-staging"

# https://google.github.io/adk-docs/deploy/agent-engine/deploy/#setup-cloud-project
# https://docs.cloud.google.com/agent-builder/agent-engine/develop/custom
# Environmental Variables for Traceing (https://docs.cloud.google.com/agent-builder/agent-engine/deploy)
# Set env variables (https://docs.cloud.google.com/agent-builder/agent-engine/manage/tracing)
# OpenTelemetry (OTel) as native standard for observability across Google Cloud: Cloud Trace, Logging, and Monitoring
  # ADK and Agent Engine are OpenTelemetry-native
  # https://docs.cloud.google.com/trace/docs/finding-traces
  # GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY) allows system to use OpenTelemetry libraries to package data and send it via OTLP endpoint
custom_env_vars = {
    "GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY": "true",
    "OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT": "true",
    "GOOGLE_CLOUD_LOCATION": "global",
}

# 1. Initialize the client
client = vertexai.Client(project="deltorobarba", location="us-central1")

RESOURCE_NAME = "projects/622694106880/locations/us-central1/reasoningEngines/8838312619447156736"


# 2. Wrap the agent
adk_app = reasoning_engines.AdkApp(
    agent=root_agent,
    enable_tracing=True,
    env_vars=custom_env_vars
)

# 3. Deploy
# remote_agent = client.agent_engines.update : for updating existing one
# remote_agent = client.agent_engines.create : to create a new one

remote_agent = client.agent_engines.update(
    name=RESOURCE_NAME,          # <-- Pflicht: identifiziert die bestehende Engine
    agent=adk_app,
    config={
        "display_name": "Agent Engine for Agent Pipeline (New)",
        "staging_bucket": STAGING_BUCKET,
        "requirements": [
            "google-cloud-aiplatform[agent_engines,adk]>=1.135",
            "google-adk>=1.23.0",
            "google-cloud-trace"
        ],
        "env_vars": custom_env_vars,
        "context_spec": {
            "memory_bank_config": {
                "generation_config": {
                    "model": f"projects/deltorobarba/locations/global/publishers/google/models/gemini-3.5-flash"
                }
            }
        }
    },
)

INFO:vertexai_genai.agentengines:Identified the following requirements: {'google-cloud-aiplatform': '1.161.0', 'pydantic': '2.13.4', 'cloudpickle': '3.1.2'}
INFO:vertexai_genai.agentengines:The following requirements are appended: {'cloudpickle==3.1.2', 'pydantic==2.13.4'}
INFO:vertexai_genai.agentengines:The final list of requirements: ['google-cloud-aiplatform[agent_engines,adk]>=1.135', 'google-adk>=1.23.0', 'google-cloud-trace', 'cloudpickle==3.1.2', 'pydantic==2.13.4']
INFO:vertexai_genai.agentengines:Using bucket deltorobarba-agent-staging
INFO:vertexai_genai.agentengines:Wrote to gs://deltorobarba-agent-staging/agent_engine/agent_engine.pkl
INFO:vertexai_genai.agentengines:Writing to gs://deltorobarba-agent-staging/agent_engine/requirements.txt
INFO:vertexai_genai.agentengines:Creating in-memory tarfile of extra_packages
INFO:vertexai_genai.agentengines:Writing to gs://deltorobarba-agent-staging/agent_engine/dependencies.tar.gz
INFO:vertexai_genai.agentengines:Using agent framew

In [3]:
# @title Test Agent ====> Single Turn Query
############################################

# 1. Initialize the client
client = vertexai.Client(project="deltorobarba", location="us-central1")

# 2. Remote app handle
RESOURCE_NAME = "projects/622694106880/locations/us-central1/reasoningEngines/8838312619447156736"
remote_app = client.agent_engines.get(name=RESOURCE_NAME)

async def run_remote_test():
    print(f"Connecting to: {RESOURCE_NAME}...")
    user_id = "deltorobarba"

    # 3. Create session and grab the 'id' key specifically 🔑
    raw_session = await remote_app.async_create_session(user_id=user_id)

    # Path A: First single-turn query:
    session_id = raw_session.get('id')
    if not session_id:
        print(f"❌ Error: ID not found in response: {raw_session}")
        return

    # Path B: Follow-up single-turn query (explicitly reuse Session ID from  first run!)
    #print(f"Resuming Session on: {RESOURCE_NAME}...")
    #session_id = "1107804831667453952"        #  <-------- copy/paste existing session ID !!!

    print(f"✅ Managed Session ID: {session_id}")

    # 4. Stream the query (nur Summarizer-Events anzeigen) 🌊
    async for event in remote_app.async_stream_query(
        user_id=user_id,
        session_id=session_id,
        #message="Uluru bank uses ChatGPT. The annual revenue of Uluru bank is 500 Mio USD. Search news on Gemini 3 and compare it to GPT-4o."
        message="I lead 100 employees. What are the latest 2026 news on Gemini 3?"
    ):
        # Filtern nach dem Author 'summarizer' 🔍
        # if 'content' in event: # print every agent's output
        if event.get('author') == 'summarizer' and 'content' in event:
            for part in event['content'].get('parts', []):
                if 'text' in part:
                    print(f"\n--- Agent Output (Summary Agent) ---\n{part['text']}")

    # 5. Trigger Memory Extraction 🧠
    print(f"\n[System] Fetching session state for extraction...")
    current_session = await remote_app.async_get_session(user_id=user_id, session_id=session_id)

    # Check for events to ensure history is captured
    # Docs: https://google.github.io/adk-docs/events/#extracting-key-information
    if current_session.get('events'):
        print(f"✅ Found {len(current_session['events'])} events. Sending to Memory Bank...")
        # Pass the dictionary directly to the memory service
        await remote_app.async_add_session_to_memory(session=current_session)
        print("\n[Done] Memory extraction triggered! Check 'Gemerkte Informationen' in 1-2 mins.")
    else:
        print("\n❌ Error: Session history was empty. No memories extracted.")

# Execute
await run_remote_test()

Connecting to: projects/622694106880/locations/us-central1/reasoningEngines/8838312619447156736...
✅ Managed Session ID: 5904387920877322240

--- Agent Output (Summary Agent) ---
Here is your executive briefing on the 2026 Gemini 3 landscape, tailored for evaluating a transition from your current OpenAI setup:

*   **Targeted 2026 Model Stratification:** Google has structured the Gemini 3 family to optimize enterprise spend. Gemini 3.8 Flash is the high-speed workhorse; 3.6 Flash slashes day-to-day API costs by requiring 17% fewer tokens for comparable results; and 3.5 Flash-Lite is built for cheap, high-volume automation. Note that updates to the high-tier 3.1 Pro are currently paused.
*   **Strong Benchmark Victories:** Gemini 3.8 Flash (competing in the 7–13B parameter class) punches well above its weight. It beats larger models like GPT-5.6 Sol and Claude Opus 5 in critical business niches: achieving **61.4% in Financial Analysis** (Vals Finance v2), **87.8% in Long Video Understan

In [ ]:
# @title Test Agent ====> Multi-Turn Query
############################################

# 1. Initialize the client
client = vertexai.Client(project="deltorobarba", location="us-central1")

# 2. Remote app handle
RESOURCE_NAME = "projects/622694106880/locations/us-central1/reasoningEngines/8838312619447156736"
remote_app = client.agent_engines.get(name=RESOURCE_NAME)


async def run_multi_turn_test():
    print(f"Connecting to: {RESOURCE_NAME}...")
    user_id = "deltorobarba"

    # --- Turn 1: Initial Query (Creates the Session) ---
    raw_session = await remote_app.async_create_session(user_id=user_id)
    session_id = raw_session.get('id')
    print(f"✅ Session Created: {session_id}")

    # FIRST MESSAGE
    await execute_query(user_id, session_id, "I am an astronaut. Uluru bank uses ChatGPT. Annual revenue is 500 Mio USD. Compare Gemini 3 to GPT-4o.")

    # --- Turn 2: Follow-up Query (Reuses the Session ID) ---
    print(f"\n--- Starting Follow-up Query in Session: {session_id} ---")

    # SECOND MESSAGE (re-uses same session_id variable)
    await execute_query(user_id, session_id, "Based on that revenue, which model is more cost-effective for us?")

    # --- Step 5: Final Memory Extraction ---
    # Fetching the session now will show events from BOTH queries.
    current_session = await remote_app.async_get_session(user_id=user_id, session_id=session_id)
    if current_session.get('events'):
        print(f"✅ Found {len(current_session['events'])} total events. Extracting to Memory Bank...")
        await remote_app.async_add_session_to_memory(session=current_session)

# Helper function to keep the code clean
async def execute_query(user_id, session_id, message):
    async for event in remote_app.async_stream_query(
        user_id=user_id,
        session_id=session_id,
        message=message
    ):
        if event.get('author') == 'summarizer' and 'content' in event:
            for part in event['content'].get('parts', []):
                if 'text' in part:
                    print(f"\n[Summarizer]: {part['text']}")

await run_multi_turn_test()

Connecting to: projects/622694106880/locations/us-central1/reasoningEngines/8838312619447156736...
✅ Session Created: 7545743998422351872

[Summarizer]: **Executive Summary: Gemini 3 vs. GPT-4o Strategy for Uluru Bank & Aerospace Operations**

*   **The Quantum Gap (Hype vs. Reality):** Google’s internal briefing heavily hypes Gemini 3's "quantum telepathy" as a groundbreaking tool for advanced physics computations. However, public benchmarks expose a stark contradiction: Gemini 3's actual Physics Reasoning (CritPt) score is a dismal **9%**. Despite the quantum marketing, it is not ready for critical aerospace telemetry.
*   **80% Lower Enterprise TCO:** For Uluru Bank’s $500M operations, migrating high-volume pipelines from GPT-4o to Gemini 3 Flash Preview slashes token costs from $2.38 to **$0.43 per 1M tokens**—drastically optimizing bottom-line API spend for your 100-employee team. 
*   **Massive Context & Software Automation:** Gemini 3 provides a **1,000,000 token context window*